In [1]:
import os
import torch
import joblib
import optuna
import numpy as np
import pandas as pd
import seaborn as sns
import torch.nn as nn
from scipy.stats import skew
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.utils import resample
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.inspection import permutation_importance
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, recall_score,precision_score, roc_auc_score
from sklearn.decomposition import PCA
import optuna
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

c:\Users\MELİSA\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
root_dir = 'smaller_dataset.csv'
df = pd.read_csv(root_dir)
df["Weight"] = df["Total Fwd Packet"] * df["Total Bwd packets"]

In [35]:
df.shape

(439447, 86)

In [ ]:
label_counts = df["Attack_Type"].value_counts()
categorical_columns = ["Src IP", 'Dst IP', "Src Port", "Dst Port", "Protocol"]  

for col in categorical_columns:
    df[col] = df[col].astype('category')

for col in ['Src IP', 'Dst IP', 'Src Port', 'Dst Port']:
    df[col + '_freq'] = df[col].map(df[col].value_counts())

df = pd.get_dummies(df, columns=['Protocol'], prefix='Proto')
df['Timestamp'] = pd.to_datetime(df['Timestamp'],  format='%d/%m/%Y %I:%M:%S %p')

attacks_to_remove = [
    "spoofing_ARP Spoofing",
    "spoofing_DNS Spoofing",
    "sqlinjection",
    "XSS",
    "Benign&Bruteforce_BruteForce",
    "Uploading_Attack"
]

df = df[~df["Attack_Type"].isin(attacks_to_remove)]

columns_to_drop = ["Flow IAT Std",
                   "Bwd Segment Size Avg",
                   "Subflow Fwd Packets",
    'Flow Duration', 
    'Subflow Bwd Packets', 
    'Fwd Packet Length Max', 
    'Fwd Packet Length Min', 
    'Flow Packets/s', 
    'Flow IAT Min', 
    'Flow IAT Max', 
    'Bwd IAT Max', 
    'Bwd IAT Min', 
    'Fwd Header Length', 
    'ACK Flag Count', 
    'Packet Length Std',
    "Fwd IAT Max",
    "Idle Max",
    "Idle Min",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Max",
    'Average Packet Size', 
    'Fwd Segment Size Avg', 
    'Fwd IAT Max', 
    'Bwd Header Length', 
    'Packet Length Mean', 
    'CWR Flag Count', 
    'Average Packet Size',
    "Flow IAT Mean",
    "Active Max",
    "Bwd Bytes/Bulk Avg",
    'Fwd IAT Mean', 
    'Active Mean', 
    'Active Std',
    "Fwd Act Data Pkts"
]

df = df.drop(columns=columns_to_drop)
df = df.drop(columns="Label")

def create_anomaly_label(row):
    if row['Attack_Type'] == 'Benign&Bruteforce_benign':  
        return 'Normal'
    else:
        return 'Attack'

df['Anomaly_Label'] = df.apply(create_anomaly_label, axis=1)

null_columns = df.isnull().sum()
null_columns = null_columns[null_columns > 0]  


inf_columns = df.columns[(df == np.inf).any() | (df == -np.inf).any()]

df = df.dropna()

df = df[~df.isin([np.inf, -np.inf]).any(axis=1)]

df.replace([np.inf, -np.inf], np.nan, inplace=True)





In [ ]:

non_numeric_df = df.select_dtypes(exclude=[np.number])


non_numeric_df.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Timestamp,Attack_Type,Proto_0,Proto_6,Proto_17,Anomaly_Label
0,192.168.137.144-192.168.137.240-49153-13217-6,192.168.137.144,49153,192.168.137.240,13217,2022-08-05 10:53:38,DoS_DoS SYN Flood,False,True,False,Attack
1,192.168.137.65-192.168.137.171-7661-6668-6,192.168.137.65,7661,192.168.137.171,6668,2022-10-26 11:53:56,DDoS_DDoS ACK Fragmentation,False,True,False,Attack
2,192.168.137.12-192.168.137.235-15376-8008-6,192.168.137.12,15376,192.168.137.235,8008,2022-10-26 15:46:03,DDoS_DDoS ACK Fragmentation,False,True,False,Attack
3,192.168.137.12-192.168.137.206-21499-55442-6,192.168.137.12,21499,192.168.137.206,55442,2022-10-26 16:50:41,DDoS_DDoS ACK Fragmentation,False,True,False,Attack
4,192.168.137.225-192.168.137.132-38616-8080-6,192.168.137.225,38616,192.168.137.132,8080,2022-09-14 11:10:19,DDoS_DDoS-HTTP Flood,False,True,False,Attack


In [44]:
# =======================
# Define Autoencoder Model
# =======================
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def get_reconstruction_error(self, x):
        output = self.forward(x)
        return F.mse_loss(output, x, reduction='none').mean(dim=1)
    

class Autoencoder2(nn.Module):
    def __init__(self, input_dim, hidden_dims, activation=nn.ReLU(), dropout_rate=0.1):
        super(Autoencoder2, self).__init__()
        
        # Build encoder layers
        encoder_layers = []
        prev_dim = input_dim
        
        for dim in hidden_dims:
            encoder_layers.append(nn.Linear(prev_dim, dim))
            encoder_layers.append(activation)
            encoder_layers.append(nn.Dropout(dropout_rate))
            prev_dim = dim
        
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Build decoder layers
        decoder_layers = []
        reversed_dims = list(reversed(hidden_dims))
        prev_dim = reversed_dims[0]
        
        for i, dim in enumerate(reversed_dims[1:], 1):
            decoder_layers.append(nn.Linear(prev_dim, dim))
            decoder_layers.append(activation)
            decoder_layers.append(nn.Dropout(dropout_rate))
            prev_dim = dim
        
        # Final output layer to reconstruct input
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        
        self.decoder = nn.Sequential(*decoder_layers)
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def encode(self, x):
        return self.encoder(x)


In [ ]:
def preprocess_data(df):

    X = df.select_dtypes(include=['number', 'bool'])
    y = df['Anomaly_Label']
    y = (y == "Attack").astype(int) 


    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

      

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    

    pca = PCA(n_components=0.95)  # Retain 95% of variance
    X_scaled = pca.fit_transform(X_scaled)
    
    return X_scaled, y

In [ ]:
def train_and_evaluate(model, train_loader, val_loader, X_test_tensor, y_val, y_test, 
                      epochs, optimizer, criterion, device, 
                      early_stopping=True, patience=5):
    
    model.to(device)
    

    model.train()
    history = {'loss': []}
    best_loss = float('inf')
    trigger_times = 0
    
    for epoch in range(epochs):
        epoch_loss = 0
        for batch in train_loader:
            inputs = batch[0].to(device)
            
     
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            
      
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        history['loss'].append(avg_loss)
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}")
        

        if epoch == 0 or (epoch+1) % 5 == 0:
            with torch.no_grad():
          
                sample_inputs = next(iter(train_loader))[0].to(device)
                sample_outputs = model(sample_inputs)
                

                print("Sample original values:", sample_inputs[0][:5].cpu().numpy())
                print("Sample reconstructed values:", sample_outputs[0][:5].cpu().numpy())
                print("Sample reconstruction error:", torch.mean((sample_inputs[0] - sample_outputs[0])**2).item())

    
        if early_stopping:
            if avg_loss < best_loss:
                best_loss = avg_loss
                best_model_state = model.state_dict()
                trigger_times = 0
            else:
                trigger_times += 1
                if trigger_times >= patience:
                    print(f"Early stopping triggered after {epoch+1} epochs")
                    break


    if early_stopping and 'best_model_state' in locals():
        model.load_state_dict(best_model_state)

 
    model.eval()
    val_errors = []
    
    with torch.no_grad():
        for batch in val_loader:
            inputs = batch[0].to(device)
            reconstructions = model(inputs)
         
            batch_errors = torch.mean((inputs - reconstructions) ** 2, dim=1).cpu().numpy()
            val_errors.extend(batch_errors)

   
    print(f"Validation errors - min: {np.min(val_errors):.6f}, max: {np.max(val_errors):.6f}, mean: {np.mean(val_errors):.6f}")
    

    best_f1 = 0
    best_threshold = 0
    
    for percentile in range(80, 100):
        threshold = np.percentile(val_errors, percentile)
        y_pred = (np.array(val_errors) > threshold).astype(int)
        f1 = f1_score(y_val, y_pred)
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    
    print(f"Optimal threshold: {best_threshold:.6f}, F1 score on validation set: {best_f1:.4f}")


    model.eval()
    with torch.no_grad():
        X_test_device = X_test_tensor.to(device)
        reconstructions = model(X_test_device)

        reconstruction_error = torch.mean((X_test_device - reconstructions) ** 2, dim=1).cpu().numpy()
    

    print(f"Test errors - min: {np.min(reconstruction_error):.6f}, max: {np.max(reconstruction_error):.6f}, mean: {np.mean(reconstruction_error):.6f}")
    

    y_test_np = np.array(y_test)
    y_pred = (reconstruction_error > best_threshold).astype(int)
    
    # Metrics
    results = {
        'threshold': best_threshold,
        'val_f1': best_f1,  # <-- Add this line
        'accuracy': accuracy_score(y_test_np, y_pred),
        'f1': f1_score(y_test_np, y_pred),
        'precision': precision_score(y_test_np, y_pred),
        'recall': recall_score(y_test_np, y_pred),
        'roc_auc': roc_auc_score(y_test_np, reconstruction_error),
        'confusion_matrix': confusion_matrix(y_test_np, y_pred),
        'reconstruction_error': reconstruction_error
    }
    
    print("\nTest Set Metrics:")
    print(f"Accuracy: {results['accuracy']:.4f}")
    print(f"F1 Score: {results['f1']:.4f}")
    print(f"Precision: {results['precision']:.4f}")
    print(f"Recall: {results['recall']:.4f}")
    print(f"ROC-AUC: {results['roc_auc']:.4f}")
    
    return model, history, results

In [ ]:
def run_hyperparameter_grid(X_train_tensor, X_val_tensor, X_test_tensor, y_val, y_test, input_dim, batch_sizes, architectures, 
                           learning_rates, epochs_list, dropout_rates, activation_functions,
                           early_stopping=True, patience=5):
    
    results_list = []
    best_f1 = 0
    best_config = None
    best_model = None
    
    total_combinations = (len(batch_sizes) * len(architectures) * len(learning_rates) * 
                         len(epochs_list) * len(dropout_rates) * len(activation_functions))
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    print(f"Running grid search with {total_combinations} combinations")
    combination_counter = 0
    
    for batch_size in batch_sizes:

        train_dataset = TensorDataset(X_train_tensor)
        val_dataset = TensorDataset(X_val_tensor)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        for architecture in architectures:
            for dropout_rate in dropout_rates:
                for activation_name, activation_fn in activation_functions.items():
                    for lr in learning_rates:
                        for epochs in epochs_list:
                        
                            combination_counter += 1
                            print(f"\nCombination {combination_counter}/{total_combinations}")
                            print(f"Testing: batch_size={batch_size}, architecture={architecture}, "
                                f"activation={activation_name}, lr={lr}, epochs={epochs}, "
                                f"dropout={dropout_rate}")
                            
                       
                            model = Autoencoder2(
                                input_dim=input_dim,
                                hidden_dims=architecture,
                                activation=activation_fn,
                                dropout_rate=dropout_rate
                            ).to(device)
                            
                        
                            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
                            criterion = nn.MSELoss()
                            
                         
                            trained_model, history, eval_results = train_and_evaluate(
                                model=model,
                                train_loader=train_loader,
                                val_loader=val_loader,
                                X_test_tensor=X_test_tensor,
                                y_val=y_val,
                                y_test=y_test,
                                epochs=epochs,
                                optimizer=optimizer,
                                criterion=criterion,
                                device=device,
                                early_stopping=early_stopping,
                                patience=patience
                            )

                      
                            config = {
                                'batch_size': batch_size,
                                'architecture': architecture,
                                'activation': activation_name,
                                'learning_rate': lr,
                                'epochs': epochs,
                                'dropout_rate': dropout_rate,
                                'early_stopping': early_stopping,
                                'patience': patience
                            }
                            
                            result = {**config, **{k: v for k, v in eval_results.items() 
                                                if k not in ['confusion_matrix', 'reconstruction_error']}}
                            results_list.append(result)
                            
                     
                            if eval_results['f1'] > best_f1:
                                best_f1 = eval_results['f1']
                                best_config = config
                                best_model = trained_model
                            
                        
                            print(f"  F1: {eval_results['f1']:.4f}, "
                                f"Precision: {eval_results['precision']:.4f}, "
                                f"Recall: {eval_results['recall']:.4f}, "
                                f"AUC: {eval_results['roc_auc']:.4f}")
                            print("-" * 50)
    
  
    results_df = pd.DataFrame(results_list)
    
    return results_df, best_model, best_config

In [ ]:
def run_full_hyperparameter_search(name, df):
 
    batch_sizes = [32]
    architectures = [
        [128, 64, 32, 16, 8],
        [16,8]
    ]
    learning_rates = [1e-3]
    epochs_list = [20]
    dropout_rates = [0.1]
    activation_functions = {
        'ReLU': nn.ReLU(),
        'LeakyReLU': nn.LeakyReLU(0.1),
        'Tanh': nn.Tanh()
    }
    

    early_stopping = True
    patience = 5

    X_scaled, y = preprocess_data(df)
    

    print(f"Data shape: {X_scaled.shape}")
    print(f"Feature stats - min: {np.min(X_scaled):.4f}, max: {np.max(X_scaled):.4f}, mean: {np.mean(X_scaled):.4f}, std: {np.std(X_scaled):.4f}")
    print(f"Label distribution: {np.bincount(y)}")
    

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X_scaled, y, test_size=0.3, random_state=42, stratify=y)
    

    X_train_full, X_val, y_train_full, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.143, random_state=42, stratify=y_train_val)  # 0.143 of 70% is ~10% of total


    X_train = X_train_full[y_train_full == 0]
    input_dim = X_train.shape[1]
    

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    

    print(f"Training tensor shape: {X_train_tensor.shape}, dtype: {X_train_tensor.dtype}")
    print(f"Validation tensor shape: {X_val_tensor.shape}, dtype: {X_val_tensor.dtype}")
    print(f"Test tensor shape: {X_test_tensor.shape}, dtype: {X_test_tensor.dtype}")
    

    y_val = y_val.values if hasattr(y_val, 'values') else y_val
    y_test = y_test.values if hasattr(y_test, 'values') else y_test
    
    results_df, best_model, best_config = run_hyperparameter_grid(
        X_train_tensor, X_val_tensor, X_test_tensor, y_val, y_test, input_dim,
        batch_sizes, architectures, learning_rates, epochs_list,
        dropout_rates, activation_functions,
        early_stopping=early_stopping, patience=patience
    )
    

    results_df.to_csv(f'{name}.csv', index=False)
    
    print("\nFull Grid Search Results Summary:")
    print(f"Number of configurations tested: {len(results_df)}")
    print("\nBest Configurations by F1 Score:")
    print(results_df.sort_values('f1', ascending=False).head())
    
    print("\nBest Configuration:")
    print(best_config)
    
    return best_model, best_config, results_df

In [ ]:
best_model, best_config, results_df = run_full_hyperparameter_search("Auto_withpca_second_attempt",df)


Data shape: (438915, 30)
Feature stats - min: -271.9800, max: 587.5274, mean: 0.0000, std: 1.1982
Label distribution: [398198  40717]
Training tensor shape: torch.Size([238878, 30]), dtype: torch.float32
Validation tensor shape: torch.Size([43936, 30]), dtype: torch.float32
Test tensor shape: torch.Size([131675, 30]), dtype: torch.float32
Using device: cuda
Running grid search with 9 combinations

Combination 1/9
Testing: batch_size=32, architecture=[128, 64, 32, 16, 8], activation=ReLU, lr=0.001, epochs=20, dropout=0.1
Epoch 1/20, Loss: 0.737775
Sample original values: [0.11769731 2.5499146  1.0878412  1.1096706  0.10272651]
Sample reconstructed values: [0.76666945 3.140007   0.83354694 0.9514036  0.07246339]
Sample reconstruction error: 0.19514136016368866
Epoch 2/20, Loss: 0.532982
Epoch 3/20, Loss: 0.490990
Epoch 4/20, Loss: 0.470124
Epoch 5/20, Loss: 0.434652
Sample original values: [ 6.5649986 -1.5380443 -1.3287997  3.5597157 -1.1576791]
Sample reconstructed values: [ 4.5582976  

In [ ]:

y_test_np = np.array(y_test)

model.eval()
with torch.no_grad():
    X_test_tensor = X_test_tensor.to(device)
    reconstructions = model(X_test_tensor)
    reconstruction_error = torch.mean((X_test_tensor - reconstructions) ** 2, dim=1).cpu().numpy()

threshold = np.percentile(reconstruction_error[y_test_np == 0], 90)
y_pred = (reconstruction_error > threshold).astype(int)


print("Threshold used:", threshold)
print("\nClassification Report:")
print(classification_report(y_test_np, y_pred, target_names=["Normal", "Anomaly"]))

print("Accuracy:", accuracy_score(y_test_np, y_pred))
print("F1 Score:", f1_score(y_test_np, y_pred))
print("Precision:", precision_score(y_test_np, y_pred))
print("Recall:", recall_score(y_test_np, y_pred))
print("ROC-AUC:", roc_auc_score(y_test_np, reconstruction_error))  


cm = confusion_matrix(y_test_np, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Anomaly"],
            yticklabels=["Normal", "Anomaly"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


'# Ensure y_test is a NumPy array\ny_test_np = np.array(y_test)\n\nmodel.eval()\nwith torch.no_grad():\n    X_test_tensor = X_test_tensor.to(device)\n    reconstructions = model(X_test_tensor)\n    reconstruction_error = torch.mean((X_test_tensor - reconstructions) ** 2, dim=1).cpu().numpy()\n\nthreshold = np.percentile(reconstruction_error[y_test_np == 0], 90)\ny_pred = (reconstruction_error > threshold).astype(int)\n\n# Compute metrics\nprint("Threshold used:", threshold)\nprint("\nClassification Report:")\nprint(classification_report(y_test_np, y_pred, target_names=["Normal", "Anomaly"]))\n\nprint("Accuracy:", accuracy_score(y_test_np, y_pred))\nprint("F1 Score:", f1_score(y_test_np, y_pred))\nprint("Precision:", precision_score(y_test_np, y_pred))\nprint("Recall:", recall_score(y_test_np, y_pred))\nprint("ROC-AUC:", roc_auc_score(y_test_np, reconstruction_error))  # continuous error values\n\n# Confusion matrix\ncm = confusion_matrix(y_test_np, y_pred)\nsns.heatmap(cm, annot=True, 